# Monte Carlo Simulation

I plan to resample the boulders from the information I have from previous competitions (this ensures a proper combination of boulders together to produce a valid prediction of the next competition's medalists).

The importance of this is because, whilst I could assign a random distribution to form a climb, a boulder with a slab score of 5, is unlikely to also have a power score of 5. Hence, I found that resampling from previous boulders to be the best way to run these simulations.

I will use the same semi-finals predicted roster, however, I will run the SF-F-Podium simulation many times, randomly resampling boulders and attaching some noise to the performance of each athlete (to simulate good/bad days) to obtain a probability distribution for the podium. We will tally how often each athlete reachs a medalling spot as well as the finals.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
events_2026 = ["KEQ26", "BER26", "MAD26", "PRA26", "IBK26"]
style_labs = ["Slab", "Coordination", "Power", "Compression", "Dynamic", "Press"]

style = pd.read_csv("BoulderStyle.csv")
style["Boulder_Num"] = style["Boulder_Num"].astype(int)

affinity = pd.read_csv("Athlete_Style_Affinity_Shrunk.csv")
affinity_cols = [f"{s}_Affinity_Shrunk" for s in style_labs]

merged = pd.read_csv("Boulder_Results_with_Style.csv")
MLdata = merged.merge(affinity, on="Athlete_ID", how="inner")

for s in style_labs:
    MLdata[f"{s}_Interaction"] = MLdata[s] * MLdata[f"{s}_Affinity_Shrunk"]
interaction_cols = [f"{s}_Interaction" for s in style_labs]
stats_cols = style_labs + affinity_cols + interaction_cols

train_df = MLdata[~MLdata["Event_ID"].isin(events_2026)].copy()

from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(train_df[stats_cols], train_df["Pct_Of_Best"])

Semifinals = pd.read_csv("SemiFinalists.csv")

In [ ]:
# Estimate the residuals to simulate the spread of the errors in our prediction model
# This allows us to include the randomness of athlete performance
train_df["Predicted_Score"] = model.predict(train_df[stats_cols])
train_df["Residual"] = train_df["Pct_Of_Best"] - train_df["Predicted_Score"]
residual_pool = train_df["Residual"].values


# -----------------------------------------------------------------------
# Generate the boulders (resample new rounds each time)

def generate_simulated_boulders(round_name, num_boulders = 4):
  pool = style[style["Round"] == round_name]       # Pool from previous boulders in that particular round
  sample = pool.sample(n = num_boulders, replace = True).reset_index(drop=True) # Allow for a boulder with the same styles (not uncommon to have multiple coordination boulders)
  return sample[style_labs]

# -----------------------------------------------------------------------
# Now predict the athletes score against the simulated boulders
def score_pred_on_boulders(roster, boulders_df, affinity_df):

  rows = [] # Dictionary to collect the athlete boulder pairings

  for boulder_idx, boulder in boulders_df.reset_index(drop = True).iterrows():
    # Loop for each boulder, and boulder_idx stores the boulder number.
    for athlete_id in roster:
      row = {"Athlete_ID": athlete_id, "Boulder_Idx": boulder_idx}

      for s in style_labs:
        row[s] = boulder[s]
      rows.append(row)

  sim_df = pd.DataFrame(rows)
  # Data frame containing all the athletes and boulders

  sim_df = sim_df.merge(affinity_df, on="Athlete_ID", how="left")
  # Add on the affinity ratings

  for s in style_labs:
    sim_df[f"{s}_Interaction"] = sim_df[s] * sim_df[f"{s}_Affinity_Shrunk"]
  # Again I require the interaction terms for the model

  sim_df["Predicted_Score"] = model.predict(sim_df[stats_cols])
  # Predict the score for each athlete

  noise = np.random.choice(residual_pool, size = len(sim_df))
  # Introducing the random noise that we established before.

  sim_df["Simulated_Score"] = np.clip(sim_df["Predicted_Score"] + noise, 0 ,1)
  # So we have our simulated score including the random noise. Ensuring we cannot leave 0 or 1.

  return sim_df


  # ---------------------------------------------------------------------------
  # Now to rank the round based on the scores

def ranks(sim_df):
  total_score = sim_df.groupby("Athlete_ID")["Simulated_Score"].sum().reset_index()
  # Group rows by athlete and sum the score together from the 4 boulders

  return total_score.sort_values("Simulated_Score", ascending=False).reset_index(drop = True)

  # --------------------------------------------------------------------------
  # Run the simulation. Semifinal round to obtain the finalists and then the ranking of the finalists

def run_one_trial(roster, affinity_df, n_finalist = 8, n_podium = 3):
  # Utilise the functions to produce the boulders and then the scores and finalists
  sf_boulders = generate_simulated_boulders("Semifinal")

  sf_scores = score_pred_on_boulders(roster, sf_boulders, affinity_df)

  sf_ranking = ranks(sf_scores)

  finalists = sf_ranking.head(n_finalist)["Athlete_ID"].tolist()

  f_boulders = generate_simulated_boulders("Final")

  f_scores = score_pred_on_boulders(finalists, f_boulders, affinity_df)

  f_ranking = ranks(f_scores)

  podium = f_ranking.head(n_podium)["Athlete_ID"].tolist()

  return finalists, podium

  # --------------------------------------------------------------------------
  # Now perform the simulation many times

roster = Semifinals["Athlete_ID"].tolist()

num_trials = 2000

finalist_counts = {a : 0 for a in roster}  # Start with 0's. Dictionary for each athlete

podium_counts = {a : 0 for a in roster}

gold_counts = {a : 0 for a in roster}

for _ in range(num_trials):     # Run the simulation 2000 times to obtain a probability distribution for each athlete
  finalists, podium = run_one_trial(roster, affinity, n_finalist = 8, n_podium = 3)

  for a in finalists:
      finalist_counts[a] += 1    # If finalist add 1

  for a in podium:
      podium_counts[a] += 1       # If athlete in podium, add 1

  if podium:
    gold_counts[podium[0]] += 1 # add 1 to gold count for the highest podium placement

results = pd.DataFrame({
    "Athlete_ID": roster,
    "Pct_Reached_Final": [finalist_counts[a] / num_trials for a in roster],
    "Pct_Podium": [podium_counts[a] / num_trials for a in roster],
    "Pct_Gold": [gold_counts[a] / num_trials for a in roster],
}).sort_values("Pct_Podium",  ascending=False)


print(results)











In [ ]:
# Reporting the %'s in a table
display_table = results.copy()
display_table["Pct_Reached_Final"] = (display_table["Pct_Reached_Final"] * 100).round(1)
display_table["Pct_Podium"] = (display_table["Pct_Podium"] * 100).round(1)
display_table["Pct_Gold"] = (display_table["Pct_Gold"] * 100).round(1)

display_table = display_table.rename(columns={
    "Pct_Reached_Final": "Reached Final (%)",
    "Pct_Podium": "Podium (%)",
    "Pct_Gold": "Gold (%)",
})

fig, ax = plt.subplots(figsize=(8, 10))
ax.axis("off")

tbl = ax.table(
    cellText=display_table.values,
    colLabels=display_table.columns,
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)

plt.tight_layout()
plt.savefig("predicted_podium_probabilities.png", dpi=200, bbox_inches="tight")
plt.show()

# Results

The model predicts that our finalists will be Sorato Anraku, Dohyun Lee, Mejdi Schalk, Meichi Narasaki, Tomoa Narasaki, Sohta Amagasa, Toby Roberts and Samuel Richard.

Currently Sorato Anraku is dominating the world cups hence the heavy bias towards him is not unusual. Though it makes it difficult to predict other athletes.